**Amelia Polak** 23362995

**Mateusz Przywozny** 23365021

**Jakub Bolka** 23378247

## 2.	The second line states if the code executes to the end without an error.

**Links to third-party implementations used for the submission:**

- https://www.geeksforgeeks.org/deep-learning/implementing-deep-q-learning-using-tensorflow/

## Assignment 2: DEEP REINFORCEMENT LEARNING

**Implementing a reinforcement learning (RL) agent using a Deep Q Network (DQN)**

**OPTION 1: Classic Control - implement DQN for one of the Gym environments for Classic Control other than the pole balancing problem.**

Using gymnasium environment, training acrobat-v1 agent

Environment -> network -> training loop -> train -> evaluate.

Then:

Adjust for double DQN -> new training setup -> train again -> evaluate and see differences

**DELETE THIS LATER**

## 1. Why Reinforcement Learning, ML paradigm of our choice

Person 1

## 2.  The Gym Environment 

Person 1

In [1]:
import gymnasium as gym
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

## 3.  Implementation 

**3.1. Capture and Sampling of the Data**

Person 1

In [2]:
# (P1: env=("Acrobot-v1"), configuration, hyperparameters, replay buffer, get_action)
state_size=6
action_size=3 
memory=0
batch_size=0
gamma=0
learning_rate=0
#change values and add more

**3.2. The Network Structure**

To train our Acrobot-v1 agent, we first used a Deep Q-Network (DQN) that learns to estimate Q-values (explained further in Section 3.3). 

In this case it is more suitable (as opposed to e.g. a Q-table) since the agent’s state is continuous and it can be in infinitely many different situations, which makes it impossible to store a Q-value for each one. That is due to its joint angles and speeds being able to take any value, e.g. 40º, 40.1º, 40.001º.

We created a class for the network which builds on the PyTorch nn.Module superclass.

For the Acrobot agent, the state has to contain six numbers and there are three possible actions for each state. The state values are cosine and sine of both joint angles of the acrobot, as well as angular velocities of both joints.
The actions are: torque left, no torque and torque right.

The network consists of: input layer with six state values -> two hidden layers with 128 neurons in each, and an output layer with three actions. (fc1, fc2, fc3)

- fc1 has 896 parameters (6 x 128 = 768 weights + 128 biases).

- fc2 has 16,512 parameters (128 x 128 = 16,384 weights + 128 biases).

- fc3 has 387 parameters (128 x 3 = 384 weights + 3 biases).

The total number of parameters in this network is 17,795.

Every time inputs pass through the network, the ReLU function is applied to remove negative values except for the final output layer which returns the list of Q-values for each action. Without ReLU, the whole network would have behaved like a one-layer pattern. ReLU makes it non-linear. The reason why it is not applied to the last layer is because Acrobot's Q-values are never positive due to the -1 reward for each step. ReLU would have turned those values into 0s.

If a GPU device is available the programme will run on it, however CPU is sufficient for a network of this size.

We made two copies of the network:

- **Working Network** - it actively learns, gets its weights updated after each step and decides which action to take.

- **Target Network** - a frozen copy that starts with identical weights as the working network and it is only updated every few steps. It is used to calculate the target values and help the working network make the right predictions and make the whole learning process more stable.

We used the Adam optimiser to train the network. It helps choose how much the weights should be adjusted after each step and takes in the specified value for learning rate. 

We also defined the loss function. We decided to go for nn.SmoothL1Loss (also known as Huber loss), instead of mean squared error. It is because early in the training, the mistakes might be quite big which MSE would square making the learning curve bounce much more. Huber loss treats small mistakes like squared error but is much more gentle on large ones (MSE on the other hand punishes large errors more) and does not get thrown off by them.

In [3]:
#Network structure for our acrobot-v1 agent
class DQN(nn.Module): #using pyTorch base class to build the Deep Q network
    def __init__(self, state_size, action_size): 
        super().__init__() #initialise parent class (the pyTorch one)
        self.fc1 = nn.Linear(state_size, 128) #input layer, state_size = 6 for acrobot -> hidden layer 1 with 128 neurons
        self.fc2 = nn.Linear(128, 128) #hidden layer 1 -> hidden layer 2 also with 128 neurons
        self.fc3 = nn.Linear(128, action_size) #hidden layer 2 -> output layer, action_size = 3 for acrobot: torque left, no torque, torque right

    def forward(self, x): #pass the state through network
        x = torch.relu(self.fc1(x)) #apply ReLU to layer 1 to remove negatives
        x = torch.relu(self.fc2(x)) #apply ReLU to layer 2
        return self.fc3(x) #return Q values for each action

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #run on gpu if available

working_network = DQN(state_size, action_size).to(device)
target_network = DQN(state_size, action_size).to(device) #make two networks, one working network and one target to help the working one learn

target_network.load_state_dict(working_network.state_dict())  #start both with identical weights

optimiser = optim.Adam(working_network.parameters(), lr=learning_rate) #use Adam optimiser to adjust weights for the working network
loss_function = nn.SmoothL1Loss() #calculate loss

**3.3. Q-Learning Update**

Q-value is the expected total reward accumulated by the agent after taking a specificic action in each state.

The whole **replay()** function is used by the network to learn using Q-values. The training loop calls the function for each step.

It takes a random number of samples of batch size that were saved in memory. If the memory does not store enough of them, the function returns None and does not proceed with training.

The sampling is random so that the steps are mixed up; using consecutive steps would have hardly made any difference and the learning would have been unstable.

Each batch consists of five parts, which are "unpacked" using the zip method.

The batch is split into states, actions, rewards, next states and "dones" (done means the action resulted in finishing the episode).

Each value is then converted into tensors and reshaped into columns using unsqueeze(1) making them match the network's output.

As for the current Q-value estimate, the working network gives three Q-values for each state. However, the only relevant one for training is the one for the action that was actually taken.

**Example:**

 Q-values: -30 (torque left), -40 (no torque), -44 (torque right) 
 
 **->** The agent ended up choosing no torque so the 'gather' function only picks out -40 and ignores the other two.

The function checks whether a plain or double DQN should be used using the "double" label. The only thing that changes is the way the new Q-value and then the target is calculated. (Check Section 6 for more details on double DQN).

For a plain DQN, the target Q-value needed for comparison and loss calculation is calculated using this formula:

**target = reward + gamma × next_q × (1 − done)**

where:

- reward is either -1 or 0 (for the final step)
- gamma is the discount factor (for Acrobot-v1 it should be around 0.99~close to 1 since there is no immediate reward coming with progress. The environment cares mostly about just reaching the line.)
- next_q is the best Q-value estimated by the target network for the next state.
- (1-done) removes the future value (next move) if the episode has ended.

The "best" new Q-value is calculated inside torch.no_grad() function which keeps the target fixed. Hence, only the network's guess gets updated and not the target.

After getting the target value, the loss function is called and Huber loss is calculated between the current prediction and the target. 

Optimiser.zero_grad() is responsible for clearing old gradients which otherwise would have been added by PyTorch to the new ones.

Loss.backward() is called to figure out how much each weight at each stage contributed to the loss (backpropagation).

Optimiser.step() updates the weights accordingly using the Adam optimiser. The target network stays as it is, only the working network gets updated.

In the end, the whole function returns the loss and the mean predicted Q-value for the batch.
These values are then logged for training and are also used to measure overestimation (Section 6).

In [4]:
def replay(double = False): #Function to learn from past experiences/actions 
    if len(memory)<batch_size: #return nothing if there is not enough saved in memory
        return None, None

    mini_batch = random.sample(memory,batch_size) #a batch of random experiences retrieved from memory

    states, actions, rewards, next_states, dones = zip (*mini_batch) #experience consist of states, actions, rewards etc. this basically unpacks the experiences
    #done means the action of that experience resulted in ending the episode

    #Every value here is turned into a tensor that can be read by the device they are sent to. (float32's & integers)
    states = torch.FloatTensor(np.array(states)).to(device) 
    actions = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(np.array(next_states)).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    current_q = working_network(states).gather(1, actions) #current predicted  Q value the network has after taking the action in each state

    with torch.no_grad(): #the target is fixed so the network can only learn by improving the Q value prediction, no_grad = no gradients
        if double: #if double DQN is used (check section 6), both networks are used to make the decision instead of just the target one
            best_next = working_network(next_states).argmax(1, keepdim = True) #the working network looks at the next actions and picks the best one. argmax returns the action number (e.g. 2), not the score
            next_q = target_network(next_states).gather(1, best_next) #then the target network gives its score for that action
        else: #plain DQN
            next_q = target_network(next_states).max(1, keepdim = True)[0] #target network picks and scores next action by taking the highest Q value which may result in overestimating (maximisation bias)

    target_q = rewards+gamma * next_q * (1-dones) #Bellman equation to get the target q value

    loss = loss_function(current_q, target_q) #calculate loss using the function defined earlier, how far the prediction is from target
    optimiser.zero_grad() #clear gradients from previous steps
    loss.backward() #Use backpropagation to figure out how much each weight contributed to the loss
    optimiser.step() #Adjust working networks weights

    return loss.item(), current_q.mean().item() #return loss and mean Q value prediction for the batch


**3.4 Training Loop**

Person 1, can also be put in 3.1 instead

## 4. Results

Person 3

## 5.  Evaluation of Results 

Person 3

## 6. Independently researched concept : Double DQN to prevent maximisation bias

Person 1 - build the training setup

Person 2 - explain maximisation bias, and double DQN, why it was used etc

Person 3 - run and evaluate

Double DQN:
best_action = argmax Q_working(next_state) then target = reward + γ × Q_target(next_state, best_action) × (1 − done)

## 7. Statement of Work

**Amelia Polak**
- contribution

**Mateusz Przywozny**
- contribution

**Jakub Bolka**
- contribution

## 8. Use of Generative AI

**Prompt:** "What is the difference between working network and target network in Reinforcement Learning?"

Answer: The working network is trained every step to predict Q-values and choose actions, while the target network is a frozen copy, refreshed only every so often, that provides stable target values so the working network isn't chasing a moving goal.

___

**Prompt:** "Demostrate an example of MSE and Huber loss handling small and large errors. Make it a table."

Answer: 
| Error | MSE | Huber |
|---|---|---|
| 0.5 | 0.25 | 0.125 |
| 10 | 100 | 9.5 |

___

**Prompt:** "What does .gather() and torch.no_grad() do in the replay function?"

Answer: .gather() keeps only the Q-value of the action actually taken; torch.no_grad() keeps the target fixed so only the network's prediction is updated.



## 9. References

- GeeksforGeeks (2019). QLearning in reinforcement learning. [online] GeeksforGeeks. Available at: https://www.geeksforgeeks.org/machine-learning/q-learning-in-python/ [Accessed 25 Nov. 2026].

- GeeksforGeeks (2019). Deep QLearning in reinforcement learning. [online] GeeksforGeeks. Available at: https://www.geeksforgeeks.org/deep-learning/deep-q-learning/ [Accessed 25 Nov. 2026].

- GeeksforGeeks (2019). Implementing deep QLearning using PyTorch. [online] GeeksforGeeks. Available at: https://www.geeksforgeeks.org/deep-learning/implementing-deep-q-learning-using-tensorflow/ [Accessed 25 Nov. 2026].

- Rose, N. (n.d.). Deep Q networks. [online] Nicholascrose.com. Available at: https://nicholascrose.com/post/ml/dqn [Accessed 25 Nov. 2026].